## 1. 환경 세팅

### 1.1 GPU 확인
- **권장**: Google Colab Pro (GPU) - 빠른 실행을 위해 권장
- Colab 사용 시: 상단 메뉴 **런타임 → 런타임 유형 변경 → GPU** 선택
- 아래 코드로 GPU가 잡혔는지 확인합니다.


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    device = "cuda"
else:
    print("GPU를 사용할 수 없습니다. CPU로 진행합니다.")
    device = "cpu"

# 이후 모델과 텐서를 이 device로 이동시킬 수 있습니다
print(f"\n사용할 device: {device}")

torch.set_printoptions(precision=4, sci_mode=False)

### 1.2 Hugging Face 실습을 위한 라이브러리
관련 라이브러리 확인 (Colab에 기본적으로 설치되어 있음)


In [ ]:
import datasets, fsspec, gcsfs, huggingface_hub, transformers
print("datasets", datasets.__version__)
print("fsspec", fsspec.__version__)
print("gcsfs", gcsfs.__version__)
print("hub", huggingface_hub.__version__)
print("transformers", transformers.__version__)

## 2. 자연어 → 토큰화 → 임베딩

### 실습 목표
- 언어를 토큰(의미 있는 단위)으로 바꾸고
- 토큰을 숫자(ID)로 바꾸고
- ID를 임베딩 벡터로 바꾸기

### 2.1 토큰화 코드 (띄어쓰기 기반)
- 실제 LLM은 BPE/SentencePiece 등 서브워드 토큰화를 사용하지만,
  원리를 이해하기 위해 **띄어쓰기 단위 토큰화**로 시작합니다.

In [ ]:
# 띄어쓰기 단위로 분리 (언어 -> 토큰)
input_text = "나는 최근 파리 여행을 다녀왔다"
input_text_list = input_text.split()
print("input_text_list: ", input_text_list)

In [ ]:
# 토큰 -> 아이디 딕셔너리와 아이디 -> 토큰 딕셔너리 만들기 (토큰 <-> 숫자)
str2idx = {word: idx for idx, word in enumerate(input_text_list)}
idx2str = {idx: word for idx, word in enumerate(input_text_list)}
print("str2idx: ", str2idx)
print("idx2str: ", idx2str)

In [ ]:
# 토큰을 토큰 아이디로 변환
input_ids = [str2idx[word] for word in input_text_list]
print("input_ids: ", input_ids)

> ✅ 체크포인트
- `input_text_list` 길이 = 시퀀스 길이(T)
- `input_ids`는 모델이 처리하는 **정수 시퀀스**입니다.

> ✍️ (미니 과제)
- 문장을 바꿔보고 토큰 리스트/ID가 어떻게 달라지는지 확인해 보세요.

In [ ]:
input_text_2 = "나는 파리 여행을 최근 다녀왔다"
input_text_2_list = input_text_2.split()

# 토큰을 토큰 아이디로 변환
input_ids_2 = [str2idx[word] for word in input_text_2_list]
print("input_ids: ", input_ids_2)

In [ ]:
input_text_3 = "나는 나는 파리 파리 여행을 다녀왔다"
input_text_3_list = input_text_3.split()

# 토큰을 토큰 아이디로 변환
input_ids_3 = [str2idx[word] for word in input_text_3_list]
print("input_ids: ", input_ids_3)

In [ ]:
input_ids_4 = [0, 2, 3, 4]

# 토큰 아이디를 토큰으로 변환
recovered_tokens = [idx2str[idx] for idx in input_ids_4]

print("input_ids:", input_ids_4)
print("복원된 토큰:", recovered_tokens)

# 토큰을 다시 문장으로 합치기
recovered_text = " ".join(recovered_tokens)
print("복원된 문장:", recovered_text)

### 2.2 토큰 아이디에서 벡터로 변환 (임베딩)
임베딩 레이어는 **(vocab_size × embedding_dim)** 파라미터를 가진 lookup table 입니다.
- 입력: 토큰 ID (정수)
- 출력: 임베딩 벡터 (실수)

In [ ]:
import torch
import torch.nn as nn

embedding_dim = 16
embed_layer = nn.Embedding(len(str2idx), embedding_dim) # (토큰 개수 x 임베딩 차원)

# 임베딩 레이어
print("임베딩 weight shape:", embed_layer.weight.shape)
print("임베딩 weight shape:", embed_layer.weight)

In [ ]:
input_ids = [0, 2, 4] # "나는 파리 다녀왔다"
input_embeddings = embed_layer(torch.tensor(input_ids))  # (T, C) = (3, 16)
input_embeddings = input_embeddings.unsqueeze(0)         # (B, T, C) = (1, 3, 16)

# 입력 문장을 임베딩한 결과
print("input_embeddings.shape:", input_embeddings.shape)
print(input_embeddings)

> ✅ 체크포인트
- 실제 모델 입력은 보통 (B, T, C) 형태를 사용합니다.
- 여기서 B=배치 크기, T=토큰 개수, C=임베딩 차원입니다.


### 2.3 입력을 임베딩으로 변환 (띄어쓰기 → ID → 임베딩)


In [ ]:
# 입력 텍스트
input_text = "나는 파리 다녀왔다"

# 1) 띄어쓰기 기반 토큰화
input_text_list = input_text.split()
print("1. 토큰:", input_text_list)

In [ ]:
# 2) 토큰 → ID 변환 (앞에서 만든 vocab 사용)
input_ids = [str2idx[word] for word in input_text_list]
print("2. 토큰 ID:", input_ids)

In [ ]:
# 3) 토큰 ID → 임베딩 벡터
input_embeddings = embed_layer(torch.tensor(input_ids))  # (T, C)
input_embeddings = input_embeddings.unsqueeze(0)         # (B, T, C)

print("3. 임베딩 shape:", input_embeddings.shape)
input_embeddings

In [ ]:
for i in range(len(input_text_list)):
  print('단어:',input_text_list[i])
  print('임베딩 값:', input_embeddings[0][i])
  print()

### 2.4 서브워드 토큰화

앞에서는 이해를 위해 띄어쓰기 기반 토큰화를 사용했습니다.
하지만 실제 LLM은 대부분 서브워드(subword) 토큰화 방식을 사용합니다.

In [ ]:
tokenizer.__len__()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased") # WordPiece

text = "나는 파리여행을다녀왔다"
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("토큰:", tokens)
print("토큰 ID:", ids)
print("토큰 개수:", len(tokens))

In [ ]:
tokenizer.__len__()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")  # SentencePiece

text = "나는 파리여행을다녀왔다"
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("토큰:", tokens)
print("토큰 ID:", ids)
print("토큰 개수:", len(tokens))

| 방식                | 대표 모델        | 특징            |
| ----------------- | ------------ | ------------- |
| **WordPiece**     | BERT         | `##`로 이어붙임 표시 |
| **BPE**           | GPT 계열       | 병합 규칙 기반      |
| **SentencePiece** | XLM-R, LLaMA | 공백도 토큰(`▁`)   |


### 2.5 Tokenizer란 무엇인가?

지금까지 우리는 입력을 다음과 같은 형태로 변환해 왔습니다.

입력 텍스트 → 토큰 → 토큰 ID → 임베딩 벡터

이 과정에서 텍스트를 토큰과 토큰 ID로 변환해 주는 역할을 하는 것이
바로 Tokenizer입니다.

(토큰 ID 를 임베딩 벡터로 변환하는 것부터는 모델의 역할)

실제로는 다음과 같은 여러 작업을 한 번에 수행합니다.

1. 토큰화(Tokenization)

    입력을 의미 있는 단위(token)로 분해

    (WordPiece / BPE / SentencePiece 등)

2. 토큰 ↔ 토큰 ID 매핑 관리

    토큰 → ID (convert_tokens_to_ids)

    ID → 토큰 (convert_ids_to_tokens)

3. 특수 토큰 처리

    [CLS], [SEP], <bos>, <eos>, <pad> 등

In [ ]:
text = "나는 파리 여행을 다녀왔다"

print("tokenize():")
tokens = tokenizer.tokenize(text)
print(tokens)

In [ ]:
# 토큰 ↔ 토큰 ID 매핑 관리

# 토큰 → 토큰 ID
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("토큰 → 토큰 ID:")
for t, i in zip(tokens, token_ids):
    print(f"{t:10s} -> {i}")

print("\n" + "-"*50)

# 토큰 ID → 토큰
recovered_tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("토큰 ID → 토큰:")
for i, t in zip(token_ids, recovered_tokens):
    print(f"{i:6d} -> {t}")

In [ ]:
# 특수 토큰 확인
print("특수 토큰들:")
print(f"BOS 토큰: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"EOS 토큰: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD 토큰: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"UNK 토큰: {tokenizer.unk_token} (ID: {tokenizer.unk_token_id})")

# 어휘 사전 크기 확인
print(f"\n어휘 사전 크기 (vocab_size): {tokenizer.vocab_size}")

In [ ]:
# 1: tokenize (텍스트 -> 토큰 리스트)
tokens = tokenizer.tokenize(text)
print("1. tokenize 결과 (토큰 리스트):", tokens)

# 2: encode (텍스트 -> ID 리스트)
input_ids = tokenizer.encode(text)
print("\n2. encode 결과 (ID 리스트):", input_ids)

# 3: __call__ (텍스트 -> 딕셔너리, return_tensors 옵션 사용 가능)
encoded = tokenizer(text, return_tensors="pt")
print("\n3. __call__ 결과 (딕셔너리):")
print(encoded)

In [ ]:
# 토큰 ID를 다시 텍스트로 변환 (디코딩)
# decode: ID 리스트 -> 텍스트
decoded_text = tokenizer.decode(input_ids, skip_special_tokens=True)
print("디코딩 결과:", decoded_text)

In [ ]:
# 배치 처리 및 패딩
texts = [
    "나는 파리 여행을 다녀왔다!​​",
    "파리는 아름다운 도시다?",
    "파리는 뷁 뷁 벩"
]

# padding=True: 짧은 시퀀스에 패딩 추가
# truncation=True: 긴 시퀀스 자르기
# max_length: 최대 길이 지정
encoded_batch = tokenizer(texts, padding=True, truncation=True,
                          max_length=20, return_tensors="pt")

print("배치 토크나이징 결과:")
print("input_ids shape:", encoded_batch["input_ids"].shape)
print("attention_mask shape:", encoded_batch["attention_mask"].shape)

print("\ninput_ids:")
print(encoded_batch["input_ids"])

print("\nattention_mask (1=실제 토큰, 0=패딩):")
print(encoded_batch["attention_mask"])

for i, text in enumerate(texts):
    ids = encoded_batch["input_ids"][i].tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    mask = encoded_batch["attention_mask"][i].tolist()

    real_tokens = [t for t, m in zip(toks, mask) if m == 1]

    print(f"[문장 {i}] {text}")
    print(f"  → 실제 토큰: {real_tokens}")

## 3. Hugging Face Transformers 문장생성

### 3.1. Hugging Face Transformers란?
- Transformer 기반 모델(BERT, GPT 등)을 간편하게 불러오고 학습·추론할 수 있는 라이브러리입니다.
- 수천 개의 사전학습된 모델이 존재하고 통일된 API로 불러와 학습, 평가, 배포까지 한 번에 가능하게 해줍니다.

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# 간단한 예시: AutoTokenizer와 AutoModel로 모델과 토크나이저를 자동으로 불러올 수 있습니다
text = "What is Huggingface Transformers? 😄 🔥🤖"

In [ ]:
# 예시 1: BERT (인코더 모델)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
bert_input = tokenizer(text, return_tensors="pt")

# BERT 토크나이저 출력 확인
print("BERT 토크나이저 출력:")
print(bert_input)
print(bert_input['input_ids'].shape)

In [ ]:
ids = bert_input["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(ids)

print("input_ids:", ids)
print("tokens:", tokens)

> ✅ 체크포인트
- `input_ids`: 토큰을 정수 ID로 변환한 결과
- `token_type_ids`: 문장 구분 (단일 문장이면 모두 0)
- `attention_mask`: 실제 토큰(1) vs 패딩(0) 구분

In [ ]:
# 한국어 모델과 토크나이저 불러오기

model_id = "Qwen/Qwen2.5-0.5B"  # 0.5B — 수업/데모용 권장


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_id)

# pad_token 설정 (경고 메시지 방지)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
model.config.pad_token_id = tokenizer.pad_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"모델: {model_id}")
print(f"토크나이저 타입: {type(tokenizer).__name__}")
print(f"모델 타입: {type(model).__name__}")
print(f"사용 device: {device}, dtype: {dtype}")

In [ ]:
import torch

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained("bert-base-uncased")
model.config.pad_token_id = tokenizer.pad_token_id
# GPU 사용 설정

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)  # 모델을 GPU로 이동

In [ ]:
# 기본 토크나이징
text = "Hello my name is James. Nice to"

# 1: encode (텍스트 -> ID 리스트)
input_ids = tokenizer.encode(text)
print("encode 결과 (ID 리스트):", input_ids)

# 2: tokenize (텍스트 -> 토큰 리스트)
tokens = tokenizer.tokenize(text)
print("tokenize 결과 (토큰 리스트):", tokens)

# 3: __call__ (텍스트 -> 딕셔너리, return_tensors 옵션 사용 가능)
encoded = tokenizer(text, return_tensors="pt")
print("\n__call__ 결과 (딕셔너리):")
print(encoded)

### 모델 구조 확인

In [ ]:
# 모델 구조 출력
print("모델 구조:")
print(model)

In [ ]:
# 모델 구성(config) 확인
print("모델 설정:")
print(f"어휘 사전 크기: {model.config.vocab_size}")
print(f"은닉층 차원: {model.config.hidden_size}")
print(f"레이어 수: {model.config.num_hidden_layers}")
print(f"어텐션 헤드 수: {model.config.num_attention_heads}")
print(f"FFN 차원: {model.config.intermediate_size}")
print(f"최대 위치 인코딩: {model.config.max_position_embeddings}")

### 문장 생성 과정

In [ ]:
prompt = "Hello my name is James. Nice to"

# 토큰화
inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

print("초기 입력 토큰:")
print(tokenizer.convert_ids_to_tokens(input_ids[0]))

In [ ]:
import torch

generated_ids = input_ids.clone()

print("\n=== 순차적 토큰 생성 시작 ===")

for step in range(10): # 10번 반복 생성
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)
    next_token_logits = outputs.logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True) # 가장 확률값이 높은 토큰 선택

    # 새 토큰 붙이기
    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)
    new_piece = tokenizer.decode(next_token_id[0], skip_special_tokens=True)
    decoded_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"\nStep {step+1}")
    print("새로 생성된 텍스트 조각:", repr(new_piece))
    print("현재까지 생성된 텍스트:")
    print(decoded_text)

In [ ]:
generated_ids = input_ids.clone()
k = 5

print("\n=== 순차적 토큰 생성 (Top-k 후보 보기) ===")

for step in range(5):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    topk_probs, topk_ids = torch.topk(probs, k, dim=-1)

    print(f"\nStep {step+1} Top-{k} 후보:")
    for p, tid in zip(topk_probs[0].tolist(), topk_ids[0].tolist()):
        piece = tokenizer.decode([tid], skip_special_tokens=True)
        print(f"  {p:.4f}  ->  {repr(piece)}  (id={tid})")

    # 최빈(가장 높은 확률) 선택
    next_id = topk_ids[:, 0:1]
    generated_ids = torch.cat([generated_ids, next_id], dim=1)

    # 지금까지 생성된 문장 출력
    current_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )
    print("→ 현재까지 생성된 문장:")
    print(current_text)

print("\n최종 텍스트:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))